In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib, os, numpy as np, pandas as pd
from scipy.stats import rankdata, pearsonr
from lightgbm import LGBMRegressor


def kl_divergence_moon_distance(X_train_moons, X_ref, feature_cols, n_bins=10):
    """
    Compute KL divergence between each training moon's feature distribution
    and the reference (test) moon's feature distribution.
    
    Lower KL = training moon is more similar to test moon.
    
    Returns dict: {moon_id: mean_kl_divergence}
    
    Method: bin each feature into n_bins quantile buckets using the reference
    distribution, then compare counts. This is stable even with tied values.
    """
    ref_arr = X_ref[feature_cols].values.astype(np.float32)
    eps = 1e-8

    # Pre-compute reference histogram per feature
    ref_hists = []
    edges_list = []
    for j in range(len(feature_cols)):
        col = ref_arr[:, j]
        # Use quantile-based edges from REFERENCE distribution
        edges = np.quantile(col, np.linspace(0, 1, n_bins + 1))
        edges[0]  -= 1e-6
        edges[-1] += 1e-6
        hist, _ = np.histogram(col, bins=edges)
        hist = hist.astype(np.float64) + eps
        hist /= hist.sum()
        ref_hists.append(hist)
        edges_list.append(edges)

    kl_per_moon = {}
    for moon, grp in X_train_moons.groupby("moon"):
        arr = grp[feature_cols].values.astype(np.float32)
        kl_vals = []
        for j in range(len(feature_cols)):
            hist_tr, _ = np.histogram(arr[:, j], bins=edges_list[j])
            hist_tr = hist_tr.astype(np.float64) + eps
            hist_tr /= hist_tr.sum()
            # KL(P_train || P_ref) — how much train diverges from ref
            kl = np.sum(hist_tr * np.log(hist_tr / ref_hists[j]))
            kl_vals.append(kl)
        kl_per_moon[moon] = float(np.mean(kl_vals))

    return kl_per_moon


def train(X_train, y_train, model_directory_path, loop_moon, embargo):

    # ── Config — tune these after seeing KL distribution ──────────────────────
    # KEEP_TOP_FRAC: fraction of training moons to keep (sorted by KL asc)
    # 0.3 = keep the 30% of moons most similar to test distribution
    # 1.0 = keep all moons (same as baseline)
    KEEP_TOP_FRAC = 0.4
    N_BINS        = 10    # histogram bins for KL computation
    N_KL_FEATS    = 50    # features to use for KL (top by variance = most informative)

    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon","id","target"]], on=["moon","id"])
    df = df.dropna(subset=["target"])

    # ── Select features for KL computation (top by variance, fast proxy) ──────
    feat_var = df[feature_cols].var().sort_values(ascending=False)
    kl_feats = feat_var.index[:N_KL_FEATS].tolist()

    # ── Compute KL divergence of each train moon vs the most recent train moons
    # We use the last 5 moons as the "reference" distribution (proxy for test)
    all_train_moons = sorted(df["moon"].unique())
    ref_moons       = all_train_moons[-5:]
    X_ref           = df[df["moon"].isin(ref_moons)]

    print(f"  Computing KL divergence ({len(all_train_moons)} train moons vs "
          f"last-{len(ref_moons)} moons as reference) ...")
    kl_scores = kl_divergence_moon_distance(df, X_ref, kl_feats, n_bins=N_BINS)

    # Sort moons by KL ascending (most similar first)
    sorted_moons = sorted(kl_scores.items(), key=lambda x: x[1])
    n_keep        = max(10, int(len(sorted_moons) * KEEP_TOP_FRAC))
    keep_moons    = [m for m, _ in sorted_moons[:n_keep]]

    print(f"  KL range: min={sorted_moons[0][1]:.4f}  max={sorted_moons[-1][1]:.4f}")
    print(f"  Keeping {n_keep}/{len(sorted_moons)} moons "
          f"(KEEP_TOP_FRAC={KEEP_TOP_FRAC})")
    print(f"  Kept moon range: {min(keep_moons)} → {max(keep_moons)}")

    # ── Train on filtered rows ─────────────────────────────────────────────────
    df_fit = df[df["moon"].isin(keep_moons)]

    df_fit = df_fit.copy()
    df_fit[feature_cols] = df_fit.groupby("moon")[feature_cols].transform(
        lambda s: s.fillna(s.median()))
    df_fit[feature_cols] = df_fit[feature_cols].fillna(0.0)
    df_fit["target_rank"] = df_fit.groupby("moon")["target"].transform(
        lambda s: (s.rank(method="average") - 1) / max(len(s)-1, 1) - 0.5)

    print(f"  Training on {len(df_fit):,} rows from {len(keep_moons)} moons ...")
    model = LGBMRegressor(
        n_estimators=200, learning_rate=0.05,
        random_state=42, n_jobs=-1, verbose=-1)
    model.fit(df_fit[feature_cols], df_fit["target_rank"])

    joblib.dump({"model": model, "features": feature_cols,
                 "keep_frac": KEEP_TOP_FRAC, "n_kept": n_keep},
                os.path.join(model_directory_path, "model.joblib"))
    print("  Saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    model    = obj["model"]
    features = obj["features"]

    X = X_test.copy()
    for c in features:
        med = X[c].median()
        X[c] = X[c].fillna(med if not np.isnan(med) else 0.0)

    raw   = model.predict(X[features])
    n     = len(raw)
    final = (rankdata(raw) - 1) / max(n-1, 1) - 0.5

    return pd.DataFrame({
        "moon": X_test["moon"].values,
        "id":   X_test["id"].values,
        "prediction": final,
    })


In [ ]:
from scipy.stats import pearsonr
import numpy as np, os
import types

# Run with KL filtering
embargo = 4

# Compare KEEP_TOP_FRAC values: 0.2, 0.4, 0.6, 1.0 (1.0 = no filtering)
for frac in [0.2, 0.4, 0.6, 1.0]:
    # Patch KEEP_TOP_FRAC dynamically via a wrapper

    # Redefine train with this frac
    def make_train(keep_frac):
        def train_kl(X_train, y_train, model_directory_path, loop_moon, embargo):

            feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]
            df = X_train.merge(y_train[["moon","id","target"]], on=["moon","id"])
            df = df.dropna(subset=["target"])

            feat_var = df[feature_cols].var().sort_values(ascending=False)
            kl_feats = feat_var.index[:50].tolist()

            all_train_moons = sorted(df["moon"].unique())
            ref_moons = all_train_moons[-5:]
            X_ref = df[df["moon"].isin(ref_moons)]

            eps = 1e-8
            ref_arr = X_ref[kl_feats].values.astype(np.float32)
            ref_hists, edges_list = [], []
            for j in range(len(kl_feats)):
                col = ref_arr[:, j]
                edges = np.quantile(col, np.linspace(0, 1, 11))
                edges[0] -= 1e-6; edges[-1] += 1e-6
                hist, _ = np.histogram(col, bins=edges)
                hist = hist.astype(np.float64) + eps; hist /= hist.sum()
                ref_hists.append(hist); edges_list.append(edges)

            kl_scores = {}
            for moon, grp in df.groupby("moon"):
                arr = grp[kl_feats].values.astype(np.float32)
                kl_vals = []
                for j in range(len(kl_feats)):
                    h, _ = np.histogram(arr[:, j], bins=edges_list[j])
                    h = h.astype(np.float64) + eps; h /= h.sum()
                    kl_vals.append(np.sum(h * np.log(h / ref_hists[j])))
                kl_scores[moon] = float(np.mean(kl_vals))

            sorted_moons = sorted(kl_scores.items(), key=lambda x: x[1])
            n_keep = max(10, int(len(sorted_moons) * keep_frac))
            keep_moons = [m for m, _ in sorted_moons[:n_keep]]

            df_fit = df[df["moon"].isin(keep_moons)].copy()
            df_fit[feature_cols] = df_fit.groupby("moon")[feature_cols].transform(
                lambda s: s.fillna(s.median()))
            df_fit[feature_cols] = df_fit[feature_cols].fillna(0.0)
            df_fit["target_rank"] = df_fit.groupby("moon")["target"].transform(
                lambda s: (s.rank(method="average") - 1) / max(len(s)-1, 1) - 0.5)

            model = LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                   random_state=42, n_jobs=-1, verbose=-1)
            model.fit(df_fit[feature_cols], df_fit["target_rank"])
            joblib.dump({"model": model, "features": feature_cols},
                        os.path.join(model_directory_path, "model.joblib"))
        return train_kl

    model_dir = f"./model_kl_{frac}"
    os.makedirs(model_dir, exist_ok=True)

    train_fn = make_train(frac)
    train_fn(X_train, y_train, model_dir, loop_moon=0, embargo=4)

    results = []
    for moon in splits["reduced_cloud"]:
        pred = infer(X_test_cloud[X_test_cloud["moon"]==moon],
                     model_dir, loop_moon=moon, embargo=4)
        results.append(pred)
    preds = pd.concat(results)

    corrs = []
    for moon in splits["reduced_cloud"]:
        merged = preds[preds["moon"]==moon].merge(
            y_test_cloud[y_test_cloud["moon"]==moon], on=["moon","id"])
        if len(merged) > 1:
            r, _ = pearsonr(merged["prediction"], merged["target"])
            corrs.append(r)
    mean_ic = float(np.mean(corrs)) if corrs else 0.0
    print(f"  KEEP_TOP_FRAC={frac:.1f}:  Mean IC = {mean_ic:+.4f}  "
          f"per-moon: {[round(c,3) for c in corrs]}")
